In [ ]:
!pip install datasets bitsandbytes catboost

In [ ]:
import pandas as pd
import wandb
wandb.init(mode="disabled")

In [ ]:
train = pd.read_parquet("/content/train.parquet")
display(train.head())

test = pd.read_csv("/content/test.csv")
display(test.head())

import joblib
train_full = joblib.load("/content/samples_prompt_train.joblib")
print(train_full[0]["prompt"])
train = pd.DataFrame(train_full)
display(train)

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


,id,content
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей..."
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...


<start_of_turn>system  
You are an AI trained to detect rhetorical manipulation in social media.  
Return ONLY the technique names from the list, comma-separated.<end_of_turn>  

<start_of_turn>user  
### Task: Identify techniques in this post using ONLY the following:  


- **loaded_language**: Emotionally charged words (positive/negative) to sway opinions. Example: “Packing Ukrainians into minibuses” evokes outrage.
- **glittering_generalities**: Vague, positive concepts (e.g., “freedom,” “unity”) to inspire emotion without substance. Example: “We are united! Our brave heroes will liberate Ukraine!”
- **euphoria:** Highlighting successes to create excitement and boost morale. Example: “Our forces crushed the enemy—total victory!”
- **appeal_to_fear**: Exaggerating threats to pressure action. Example: “Unregistered Ukrainians abroad will lose banking access.”
- **fud**: Spreading doubt or vague threats to destabilize trust. Example: “Was Zelensky really in Bakhmut? No one can know for

,content,techniques,examples_texts,examples_techniques,prompt,output
0,Новий огляд мапи DeepState від російського вій...,"[euphoria, loaded_language]",[⚡️\nПремʼєра! Четвертий випуск програми «Реал...,"[[loaded_language], [loaded_language, appeal_t...",<start_of_turn>system \nYou are an AI trained...,"euphoria, loaded_language"
1,Недавно 95 квартал жёстко поглумился над русск...,"[loaded_language, cherry_picking]",[🤦‍♂️\n Представитель властей европейской стол...,"[[loaded_language, cherry_picking], [loaded_la...",<start_of_turn>system \nYou are an AI trained...,"loaded_language, cherry_picking"
2,🤩\nТим часом йде евакуація Бєлгородського авто...,"[loaded_language, euphoria]","[❗️\nзахарова заявила, що за терактом у Бєлгор...","[[loaded_language, cliche], [loaded_language],...",<start_of_turn>system \nYou are an AI trained...,"loaded_language, euphoria"
3,В Україні найближчим часом мають намір посилит...,[],[❗️\nПосилення кримінальної відповідальності з...,"[[glittering_generalities, loaded_language], [...",<start_of_turn>system \nYou are an AI trained...,
4,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",[loaded_language],[🔥\n🔥\n72 омсбр рф розбита вщент!\n З'явилися ...,"[[euphoria], [loaded_language, euphoria], [cli...",<start_of_turn>system \nYou are an AI trained...,loaded_language
...,...,...,...,...,...,...
3817,🤭\nросія ставить ППО на дахах адмінбудівель\nр...,"[loaded_language, euphoria]",[Тварини почали свій ранок з обстрілу критично...,"[[loaded_language, appeal_to_fear], [cherry_pi...",<start_of_turn>system \nYou are an AI trained...,"loaded_language, euphoria"
3818,"К слову, Бабий не просто «ларечник», а и челов...",[loaded_language],"[Уже восемнадцатый день, как друг и бизнес-пар...","[[fud], [loaded_language, fud, cherry_picking]...",<start_of_turn>system \nYou are an AI trained...,loaded_language
3819,"Глава ФСБ Бортников ответил журналистам, почем...",[],"[""Мы не можем наступать"". В Украине заподозрил...","[[fud], [loaded_language], [fud], [glittering_...",<start_of_turn>system \nYou are an AI trained...,
3820,В ДНР завозили наркотики в бытовой технике\nОб...,[],[❗️\nЖурналист погиб в центре Донецка при обст...,"[[cherry_picking], [loaded_language, appeal_to...",<start_of_turn>system \nYou are an AI trained...,


In [ ]:
# Make the huggingface dataset from train:

# 1. For each label in techniques crete a column:
from collections import Counter

unique_techniques = Counter()
for t in train["techniques"]:
    unique_techniques.update(t)

techniques_dict = {k: i for i, k in enumerate(unique_techniques.keys())}

new_column_values = []
for t in train["techniques"]:
    if t is None:
        new_column_values.append([])
    else:
        new_column_values.append(t)
train["techniques"] = new_column_values

for k in techniques_dict.keys():
    train[k] = train["techniques"].apply(lambda x: k in x)

display(train.head())

,content,techniques,examples_texts,examples_techniques,prompt,output,euphoria,loaded_language,cherry_picking,glittering_generalities,cliche,appeal_to_fear,bandwagon,fud,whataboutism,straw_man
0,Новий огляд мапи DeepState від російського вій...,"[euphoria, loaded_language]",[⚡️\nПремʼєра! Четвертий випуск програми «Реал...,"[[loaded_language], [loaded_language, appeal_t...",<start_of_turn>system \nYou are an AI trained...,"euphoria, loaded_language",True,True,False,False,False,False,False,False,False,False
1,Недавно 95 квартал жёстко поглумился над русск...,"[loaded_language, cherry_picking]",[🤦‍♂️\n Представитель властей европейской стол...,"[[loaded_language, cherry_picking], [loaded_la...",<start_of_turn>system \nYou are an AI trained...,"loaded_language, cherry_picking",False,True,True,False,False,False,False,False,False,False
2,🤩\nТим часом йде евакуація Бєлгородського авто...,"[loaded_language, euphoria]","[❗️\nзахарова заявила, що за терактом у Бєлгор...","[[loaded_language, cliche], [loaded_language],...",<start_of_turn>system \nYou are an AI trained...,"loaded_language, euphoria",True,True,False,False,False,False,False,False,False,False
3,В Україні найближчим часом мають намір посилит...,[],[❗️\nПосилення кримінальної відповідальності з...,"[[glittering_generalities, loaded_language], [...",<start_of_turn>system \nYou are an AI trained...,,False,False,False,False,False,False,False,False,False,False
4,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",[loaded_language],[🔥\n🔥\n72 омсбр рф розбита вщент!\n З'явилися ...,"[[euphoria], [loaded_language, euphoria], [cli...",<start_of_turn>system \nYou are an AI trained...,loaded_language,False,True,False,False,False,False,False,False,False,False


In [ ]:
import copy
print(train[unique_techniques.keys()].mean())
additional_train = train[(train.straw_man | train.whataboutism | train.bandwagon) & ~train.loaded_language]

additional_train_full = []

for _, row in additional_train.iterrows():
  if (row["techniques"] is not None) and len(row["techniques"]) > 1:
    for t in row["techniques"]:
      row_copy = copy.deepcopy(row)
      row_copy[unique_techniques.keys()] = [True if a == t else False for a in list(unique_techniques.keys())]
      additional_train_full.append(row_copy.to_dict())

additional_train_full = pd.DataFrame(additional_train_full)
additional_train_full = pd.concat([additional_train_full, additional_train])
len(additional_train_full)

euphoria                   0.120879
loaded_language            0.516222
cherry_picking             0.133961
glittering_generalities    0.126374
cliche                     0.121141
appeal_to_fear             0.078493
bandwagon                  0.041078
fud                        0.100733
whataboutism               0.041340
straw_man                  0.036107
dtype: float64


279

In [ ]:
# 2. Create a huggingface dataset:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train)
display(train_dataset)

Dataset({
    features: ['content', 'techniques', 'examples_texts', 'examples_techniques', 'prompt', 'output', 'euphoria', 'loaded_language', 'cherry_picking', 'glittering_generalities', 'cliche', 'appeal_to_fear', 'bandwagon', 'fud', 'whataboutism', 'straw_man'],
    num_rows: 3822
})

In [ ]:
additional_train = Dataset.from_pandas(additional_train_full)

In [ ]:
labels = list(techniques_dict.keys())
id2label = {idx:label for idx, label in enumerate(labels)}
label2id = {label:idx for idx, label in enumerate(labels)}
labels

['euphoria',
 'loaded_language',
 'cherry_picking',
 'glittering_generalities',
 'cliche',
 'appeal_to_fear',
 'bandwagon',
 'fud',
 'whataboutism',
 'straw_man']

In [ ]:
MODEL_NAME = "google/gemma-2-2b-it"

In [ ]:
from transformers import AutoTokenizer
import numpy as np
from datasets import DatasetDict, concatenate_datasets
access_token = "XXX"  # Replace with your actual access token

MAX_LENGTH = 3000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token = access_token) #, padding_side="right")
tokenizer.pad_token = tokenizer.eos_token  # Required for Gemma
tokenizer.pad_token_id = tokenizer.eos_token_id

def preprocess_data(examples):
  # take a batch of texts
  text = examples["prompt"]
  # encode them
  # encoding = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LENGTH, return_tensors="np")
  encoding = tokenizer(text, padding="longest", truncation=True, max_length=MAX_LENGTH, return_tensors="np")
  # add labels
  labels_batch = {k: examples[k] for k in examples.keys() if k in labels}
  # create numpy array of shape (batch_size, num_labels)
  labels_matrix = np.zeros((len(text), len(labels)))
  # fill numpy array
  for idx, label in enumerate(labels):
    labels_matrix[:, idx] = labels_batch[label]

  encoding["labels"] = labels_matrix.tolist()

  return encoding


train_test = train_dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)

final_splits = {
    "train": train_test["train"],
    "validation": test_valid["train"],  # 10% of original dataset
    "test": test_valid["test"]  # 10% of original dataset
}
train_dataset = DatasetDict({
    "train": final_splits["train"],
    "validation": final_splits["validation"],
    "test": final_splits["test"]
})

encoded_dataset = train_dataset.map(preprocess_data, batched=True, remove_columns=train_dataset["train"].column_names, batch_size=1)  # Remove before tensorization) #, remove_columns=dataset['train'].column_names)

encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_dataset


Map:   0%|          | 0/3057 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3057
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 382
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 383
    })
})

In [ ]:
from transformers import AutoModelForSequenceClassification, Gemma2ForSequenceClassification
from transformers import AutoModel, AutoConfig, BitsAndBytesConfig
from typing import Optional
import torch

config = AutoConfig.from_pretrained(MODEL_NAME, token=access_token)
config.update({
    ####### include in appendix
    "hidden_dropout_prob": 0.2,  # Default 0.1
    "attention_probs_dropout_prob": 0.2,  # Default 0.1
    "classifier_dropout": 0.3,  # Add explicit classifier dropout    --> parameters to avoid overfitting
    ########
    "problem_type": "multi_label_classification",
    "num_labels": len(labels),
    "id2label": id2label,
    "label2id": label2id,
    "ignore_mismatched_sizes": True,
    "token": access_token,
})

from peft import PrefixTuningConfig, LoraConfig, get_peft_model

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    token= access_token,
)

base_model.config.pad_token_id = tokenizer.pad_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at google/gemma-2-2b-it and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "/content/adapter")
model = model.merge_and_unload()

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['corda_config', 'trainable_token_indices'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [ ]:
batch_size = 1
metric_name = "f1"

from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch
import warnings
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

args = TrainingArguments(
    f"roberta_large",
    evaluation_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    gradient_accumulation_steps=6,
    save_steps = 100,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    # load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    #push_to_hub=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,  # Reduces memory issues,
    label_names=["labels"],
)

def get_f1_macro_thr_cv_stratified(targets, outputs, n_splits=5):
    num_targets = targets.shape[1]
    avg_thresholds = np.zeros(num_targets)

    # Ignore warnings (useful for cases where f1_score might issue warnings)
    warnings.filterwarnings("ignore")

    for i in range(num_targets):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        thresholds = []

        for train_idx, val_idx in skf.split(np.zeros(len(targets)), targets[:, i]):
            val_targets, val_outputs = targets[val_idx, i], outputs[val_idx, i]
            best_f1, best_thresh = 0, 0.01

            for threshold in np.arange(0.01, 0.99, 0.01):
                preds = (val_outputs > threshold).astype(int)
                f1 = f1_score(val_targets, preds)
                if f1 > best_f1:
                    best_f1, best_thresh = f1, threshold

            thresholds.append(best_thresh)
        # print("All thresholds: ", thresholds)
        avg_thresholds[i] = np.median(thresholds)  # Average thresholds across folds

    # Compute final validation predictions using averaged thresholds
    validation_predictions = (outputs > avg_thresholds).astype(int)
    final_f1 = f1_score(targets, validation_predictions, average='macro')

    print("Stable averaged thresholds across folds:", avg_thresholds)
    print("Final F1-macro:", final_f1)

    return avg_thresholds, final_f1


def multi_label_metrics(predictions, labels):
    outputs = torch.sigmoid(torch.tensor(predictions)).cpu().numpy()
    f1 = get_f1_macro_thr_cv_stratified(labels, outputs)[1]
    roc_auc = roc_auc_score(labels, outputs, average = 'macro')
    metrics = {'f1': f1,
               'roc_auc': roc_auc}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions,
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.2,
    bias="none",
    modules_to_save=["classifier", "score"],  # <--- Preserve classification head
    task_type="SEQ_CLS"  # <--- Not CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,789,760 || all params: 2,635,154,688 || trainable%: 0.7889


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    max_length=MAX_LENGTH,
    return_tensors="pt",
    )

trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

<ipython-input-21-6812eaddae9f>:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

# Select thresholds

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from peft import PeftModel
from transformers import pipeline
from transformers import AutoTokenizer
import numpy as np
from datasets import DatasetDict, concatenate_datasets
access_token = "XXX"

config = AutoConfig.from_pretrained(MODEL_NAME)
config.update({
    "hidden_dropout_prob": 0.2,  # Default 0.1
    "attention_probs_dropout_prob": 0.2,  # Default 0.1
    "classifier_dropout": 0.3,  # Add explicit classifier dropout
    "problem_type": "multi_label_classification",
    "num_labels": len(labels),
    "id2label": id2label,
    "label2id": label2id,
    "ignore_mismatched_sizes": True,
    "token": access_token,
})

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    # quantization_config= bnb_config
)
base_model.config.pad_token_id = tokenizer.pad_token_id

train_dataset = Dataset.from_pandas(train)
train_test = train_dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)
final_splits = {
    "train": train_test["train"],
    "validation": test_valid["train"],  # 10% of original dataset
    "test": test_valid["test"]  # 10% of original dataset
}
train_dataset = DatasetDict({
    "train": final_splits["train"],
    "validation": final_splits["validation"],
    "test": final_splits["test"]
})

encoded_dataset = train_dataset.map(preprocess_data, batched=True)  # Remove before tensorization) #, remove_columns=dataset['train'].column_names)

encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_dataset

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at google/gemma-2-2b-it and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/3057 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['content', 'techniques', 'examples_texts', 'examples_techniques', 'prompt', 'output', 'euphoria', 'loaded_language', 'cherry_picking', 'glittering_generalities', 'cliche', 'appeal_to_fear', 'bandwagon', 'fud', 'whataboutism', 'straw_man', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3057
    })
    validation: Dataset({
        features: ['content', 'techniques', 'examples_texts', 'examples_techniques', 'prompt', 'output', 'euphoria', 'loaded_language', 'cherry_picking', 'glittering_generalities', 'cliche', 'appeal_to_fear', 'bandwagon', 'fud', 'whataboutism', 'straw_man', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 382
    })
    test: Dataset({
        features: ['content', 'techniques', 'examples_texts', 'examples_techniques', 'prompt', 'output', 'euphoria', 'loaded_language', 'cherry_picking', 'glittering_generalities', 'cliche', 'appeal_to_fear', 'bandwagon', 'fud', 'whataboutism', 'straw_man', 'i

In [ ]:
model = PeftModel.from_pretrained(base_model, "/content/adapter")
model = model.merge_and_unload()

model = PeftModel.from_pretrained(model, "/content/roberta_large/checkpoint-1000")
model = model.merge_and_unload()

In [ ]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [ ]:
pred = classifier(encoded_dataset["validation"].to_pandas().prompt.to_list(), return_all_scores=True)
pred_scores_val = np.array([[i["score"] for i in j] for j in pred])
true_labels_val = np.array(encoded_dataset["validation"].to_pandas().labels.to_list())
best_threshold, _ = get_f1_macro_thr_cv_stratified(true_labels_val, pred_scores_val)

In [ ]:
pred_ = classifier(encoded_dataset["test"].to_pandas().prompt.to_list(), return_all_scores=True)
pred_scores_test_ = np.array([[i["score"] for i in j] for j in pred_])
true_labels_test_ = np.array(encoded_dataset["test"].to_pandas().labels.to_list())
best_threshold, _ = get_f1_macro_thr_cv_stratified(true_labels_test_, pred_scores_test_)

Stable averaged thresholds across folds: [0.21 0.26 0.31 0.44 0.14 0.23 0.12 0.43 0.05 0.21]
Final F1-macro: 0.48738925437001734


In [ ]:
pred_all = np.concat([pred_scores_val, pred_scores_test_])
true_all = np.concat([true_labels_val, true_labels_test_])
best_threshold, _ = get_f1_macro_thr_cv_stratified(true_all, pred_all)

Stable averaged thresholds across folds: [0.21 0.38 0.28 0.5  0.12 0.27 0.18 0.45 0.05 0.18]
Final F1-macro: 0.4792496286136547


In [ ]:
pred_ = classifier(encoded_dataset["train"].to_pandas().prompt.to_list(), return_all_scores=True)
pred_scores_train = np.array([[i["score"] for i in j] for j in pred_])
true_labels_train = np.array(encoded_dataset["train"].to_pandas().labels.to_list())

# Make the final prediction

In [ ]:
test_df = pd.DataFrame(joblib.load("/content/samples_prompt_test.joblib"))
test_df.head()

,content,techniques,examples_texts,examples_techniques,prompt,output
0,"Они просрали нашу технику, положили кучу людей...",[],"[Україна: ""Дайте вже нарешті ці F-16, нам дуже...","[[loaded_language], [loaded_language, appeal_t...",<start_of_turn>system \nYou are an AI trained...,
1,❗️\nКитай предлагает отдать оккупированные тер...,[],[❗️\nВсе населенные пункты Харьковской области...,"[[loaded_language, euphoria], [loaded_language...",<start_of_turn>system \nYou are an AI trained...,
2,Сегодня будет ровно 6 месяцев с этого обещания...,[],[Ойойой. Це зальот. Якщо це розійдеться погана...,"[[loaded_language, appeal_to_fear], [loaded_la...",<start_of_turn>system \nYou are an AI trained...,
3,⚡️\nІзраїль вперше у світі збив балістичну рак...,[],[Пока большинство стран мира призывают прекрат...,"[[loaded_language], [loaded_language, appeal_t...",<start_of_turn>system \nYou are an AI trained...,
4,Склав невелику навчально-методичну таблицю на ...,[],"[Гюнтер Фелінгер, економіст, голова Австрійськ...","[[loaded_language, fud], [loaded_language], [l...",<start_of_turn>system \nYou are an AI trained...,


In [ ]:
preds_test = classifier(test_df.prompt.to_list(), return_all_scores=True)

In [ ]:
# best_thresholds = np.array([0.3, 0.13, 0.21, 0.21, 0.18, 0.22, 0.15, 0.32, 0.1, 0.09])
test_df_for_id = pd.read_csv("/content/test.csv")
best_thresholds = best_threshold
pred_scores_test = np.array([[i["score"] for i in j] for j in preds_test])
pred_scores_binary = (pred_scores_test > best_thresholds).astype(int)
pred_labels = [i["label"] for i in preds_test[0]]

pred_dataframe = pd.DataFrame(pred_scores_binary, columns=pred_labels)
pred_dataframe["id"] = test_df_for_id["id"].values
pred_dataframe

,euphoria,loaded_language,cherry_picking,glittering_generalities,cliche,appeal_to_fear,bandwagon,fud,whataboutism,straw_man,id
0,0,1,0,0,1,0,0,1,0,0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba
1,0,1,0,0,0,0,0,0,0,0,9b2a61e4-d14e-4ff7-b304-e73d720319bf
2,0,1,0,0,0,0,0,0,0,0,f0f1c236-80a8-4d25-b30c-a420a39be632
3,0,0,0,0,0,0,0,0,0,0,31ea05ba-2c2b-4b84-aba7-f3cf6841b204
4,0,1,0,0,1,0,0,0,1,0,a79e13ec-6d9a-40b5-b54c-7f4f743a7525
...,...,...,...,...,...,...,...,...,...,...,...
5730,0,1,1,0,1,0,0,0,0,0,e8e22b6d-0068-4afb-b606-4a1baa8a8d4c
5731,0,1,1,0,0,0,0,1,0,0,8b1d69b4-69ce-4e40-b4ba-dd2f370a8b6f
5732,0,1,0,0,0,0,0,0,0,0,c2246217-3358-4f61-bda8-e2ec21aed5b2
5733,1,0,0,0,0,0,0,0,0,0,45aa63c4-2248-4a0e-8f66-f3d23b6828ed


In [ ]:
pred_dataframe[unique_techniques.keys()].mean()

,0
euphoria,0.111944
loaded_language,0.605231
cherry_picking,0.143156
glittering_generalities,0.112293
cliche,0.220052
appeal_to_fear,0.115083
bandwagon,0.025283
fud,0.113688
whataboutism,0.099041
straw_man,0.059983


In [ ]:
pred_dataframe.to_csv("gemma_LM_CLS.csv", index=False)

 # Working with thresholds (addtional model on top with metafeatures from baseline):

In [ ]:
additional_features_train = np.load("/content/sim_techniques_vectors.npy")
additional_features_test = np.load("/content/sim_techniques_vectors_test.npy")

print(additional_features_train.shape)
print(additional_features_test.shape)

# fillna -1:
additional_features_train__ = np.nan_to_num(additional_features_train, nan=0)
additional_features_test__ = np.nan_to_num(additional_features_test, nan=0)

(3822, 45)
(5735, 45)


In [ ]:
train_content_to_idx = {t: idd for idd, t in enumerate(train.content.values)}
additional_features_train = np.array([additional_features_train__[train_content_to_idx[c]] for c in encoded_dataset["train"].to_pandas().content.to_list()])
additional_features_test = np.array([additional_features_train__[train_content_to_idx[c]] for c in encoded_dataset["test"].to_pandas().content.to_list()])
additional_features_val = np.array([additional_features_train__[train_content_to_idx[c]] for c in encoded_dataset["validation"].to_pandas().content.to_list()])

In [ ]:
all_features_train = np.concatenate([pred_scores_train, additional_features_train], axis=1)
all_features_test = np.concatenate([pred_scores_test_, additional_features_test], axis=1)
all_features_val = np.concatenate([pred_scores_val, additional_features_val], axis=1)

all_features_test_ = np.concatenate([pred_scores_test, additional_features_test__], axis=1)


In [ ]:
import joblib
# joblib.dump([all_features_train, all_features_test, all_features_val, all_features_test_], "/content/features_dump_gemma.joblib")
all_features_train, all_features_test, all_features_val, all_features_test_ = joblib.load("/content/features_dump_gemma.joblib")

In [ ]:
# Building binary classifier for each technique:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score, roc_auc_score

cat_features = []

train_data = Pool(
    data=all_features_train,
    label=true_labels_train,
    cat_features=cat_features
)
val_data = Pool(
    data=all_features_val,
    label=true_labels_val,
    cat_features=cat_features,
)
test_data = Pool(
    data=all_features_test,
    label=true_labels_test_,
    cat_features=cat_features,
)

# Multilabel classifier:
model_cat = CatBoostClassifier(
    iterations=10000,
    loss_function='MultiLogloss',
    learning_rate=0.001,
    verbose=50,
    depth=2,  # to prevent overfitting
    colsample_bylevel=0.1,  # to prevent overfitting
    subsample=0.7, bootstrap_type='Bernoulli',  # to prevent overfitting
    random_seed=42,
    # auto_class_weights='Balanced'
)

model_cat.fit(train_data, eval_set=test_data, plot=False, verbose=200, metric_period=10, use_best_model=True)

0:	learn: 0.6922219	test: 0.6921999	best: 0.6921999 (0)	total: 10.4ms	remaining: 1m 44s
200:	learn: 0.5224751	test: 0.5206061	best: 0.5206061 (200)	total: 2.01s	remaining: 1m 37s
400:	learn: 0.4214588	test: 0.4185120	best: 0.4185120 (400)	total: 4.04s	remaining: 1m 36s
600:	learn: 0.3618691	test: 0.3583573	best: 0.3583573 (600)	total: 6.08s	remaining: 1m 35s
800:	learn: 0.3244081	test: 0.3209945	best: 0.3209945 (800)	total: 8.14s	remaining: 1m 33s
1000:	learn: 0.2991257	test: 0.2963027	best: 0.2963027 (1000)	total: 10.2s	remaining: 1m 31s
1200:	learn: 0.2817152	test: 0.2797308	best: 0.2797308 (1200)	total: 12.4s	remaining: 1m 30s
1400:	learn: 0.2692464	test: 0.2684663	best: 0.2684663 (1400)	total: 14.6s	remaining: 1m 29s
1600:	learn: 0.2598502	test: 0.2603199	best: 0.2603199 (1600)	total: 16.7s	remaining: 1m 27s
1800:	learn: 0.2527826	test: 0.2543936	best: 0.2543936 (1800)	total: 18.8s	remaining: 1m 25s
2000:	learn: 0.2467785	test: 0.2497185	best: 0.2497185 (2000)	total: 20.9s	remainin

In [ ]:
y_pred_probs = model_cat.predict_proba(test_data)
class_thresholds_, metric = get_f1_macro_thr_cv_stratified(true_labels_test_, y_pred_probs, n_splits=5)

Stable averaged thresholds across folds: [0.16 0.25 0.28 0.44 0.24 0.14 0.13 0.31 0.18 0.15]
Final F1-macro: 0.4760559317833211


In [ ]:
y_pred_probs = model_cat.predict_proba(val_data)
class_thresholds_, metric = get_f1_macro_thr_cv_stratified(true_labels_val, y_pred_probs, n_splits=5)

Stable averaged thresholds across folds: [0.31 0.61 0.31 0.35 0.14 0.21 0.11 0.18 0.12 0.13]
Final F1-macro: 0.48176101734042803


In [ ]:
y_pred_probs_t = model_cat.predict_proba(test_data)
y_pred_probs_v = model_cat.predict_proba(val_data)
class_thresholds_, metric = \
  get_f1_macro_thr_cv_stratified(
      np.concatenate([true_labels_test_, true_labels_val]),
      np.concatenate([y_pred_probs_t, y_pred_probs_v]), n_splits=5)


Stable averaged thresholds across folds: [0.29 0.29 0.31 0.35 0.16 0.23 0.12 0.15 0.16 0.14]
Final F1-macro: 0.4816162898032063


In [ ]:
test_df = pd.DataFrame(joblib.load("/content/samples_prompt_test.joblib"))
preds_test_helper = classifier([test_df.prompt.to_list()[0]], return_all_scores=True)


pred_scores_test = model_cat.predict_proba(all_features_test_)
test_df_for_id = pd.read_csv("/content/test.csv")
pred_scores_binary = (pred_scores_test > class_thresholds_).astype(int)
pred_labels = [i["label"] for i in preds_test_helper[0]]

pred_dataframe = pd.DataFrame(pred_scores_binary, columns=pred_labels)
pred_dataframe["id"] = test_df_for_id["id"].values
display(pred_dataframe[unique_techniques.keys()].mean())
pred_dataframe.to_csv("gemma_LM_CLS_advanced.csv", index=False)
pred_dataframe

,0
euphoria,0.120837
loaded_language,0.631212
cherry_picking,0.140192
glittering_generalities,0.121360
cliche,0.290846
appeal_to_fear,0.076024
bandwagon,0.059459
fud,0.164778
whataboutism,0.055623
straw_man,0.073235


,euphoria,loaded_language,cherry_picking,glittering_generalities,cliche,appeal_to_fear,bandwagon,fud,whataboutism,straw_man,id
0,0,1,0,0,1,0,0,1,0,0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba
1,0,1,0,0,0,0,0,0,0,0,9b2a61e4-d14e-4ff7-b304-e73d720319bf
2,0,1,0,0,0,0,0,0,0,0,f0f1c236-80a8-4d25-b30c-a420a39be632
3,0,0,0,0,0,0,0,0,0,0,31ea05ba-2c2b-4b84-aba7-f3cf6841b204
4,0,1,0,0,1,0,1,0,0,0,a79e13ec-6d9a-40b5-b54c-7f4f743a7525
...,...,...,...,...,...,...,...,...,...,...,...
5730,0,1,1,0,1,0,1,1,0,1,e8e22b6d-0068-4afb-b606-4a1baa8a8d4c
5731,0,1,1,0,0,0,0,1,0,0,8b1d69b4-69ce-4e40-b4ba-dd2f370a8b6f
5732,0,1,0,0,0,0,0,0,0,0,c2246217-3358-4f61-bda8-e2ec21aed5b2
5733,0,0,0,0,0,0,0,0,0,0,45aa63c4-2248-4a0e-8f66-f3d23b6828ed
